# 条件随机场 (CRF) 与结构化预测算法企业笔试手撕通关宝典
> **面向对象**：互联网大厂/AI 独角兽企业 NLP 算法岗、信息抽取/NER/序列标注面试手撕  
> **核心涵盖**：发射得分、转移矩阵、目标金标路径打分、前向 Log-Sum-Exp 规范化因子、CRF 负对数似然损失、维特比最优路径解码、变长 Mask 批处理、BiLSTM-CRF 端到端组装、MEMM 标注偏置陷阱复现  
> **设计准则**：纯 PyTorch / NumPy 极简实现，零第三方 CRF 库依赖，彻底掌握全局归一化与动态规划时空开销。

---
### 核心模块速览
1. **模块一**：发射得分矩阵 (Emission Matrix) 与标签体系设计
2. **模块二**：转移矩阵 (Transition Matrix) 与非法转移硬掩码约束
3. **模块三**：金标真实路径得分 (Gold Path Score) 闭式向量化计算
4. **模块四**：前向算法与规范化因子 $\log Z(x)$ (纯向量化 Log-Sum-Exp 防溢出)
5. **模块五**：CRF 负对数似然损失函数 (CRF NLL Loss) 手撕
6. **模块六**：维特比动态规划全局最优解码 (Viterbi Decoding with Backpointers)
7. **模块七**：工业级批处理与变长 Padding Mask 机制
8. **模块八**：BiLSTM-CRF 极简端到端序列标注模型封装
9. **模块九**：MEMM 局部归一化“标注偏置 (Label Bias)”物理复现与 CRF 破局验证

---
## 模块一：发射得分矩阵 (Emission Matrix) 与标签体系设计

### 【笔试考点与陷阱】
1. **发射分数 (Emission Score)**：由底层编码器（如 BiLSTM 或 BERT）提供，形状为 `(seq_len, num_tags)`，第 $i$ 个词预测为第 $j$ 种标签的无约束原始打分；
2. **特殊边界标签**：标准 CRF 必须包含 `<START>` 与 `<STOP>` 两个虚拟边界标签，确保序列起止符合语法规则。

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 定义标签系统 (BIO 标注体系 + 起始/终止虚拟标签)
TAG_TO_IDX = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-LOC": 3,
    "I-LOC": 4,
    "<START>": 5,
    "<STOP>": 6
}
NUM_TAGS = len(TAG_TO_IDX)
START_TAG = "<START>"
STOP_TAG = "<STOP>"

print(f"标签字典大小: {NUM_TAGS}")
print("标签字典映射:", TAG_TO_IDX)

---
## 模块二：转移矩阵 (Transition Matrix) 与非法转移硬掩码约束

### 【笔试考点与陷阱】
1. **转移矩阵定义**：`transitions[i, j]` 表示从标签 $j$ 转移到标签 $i$（或从 $i$ 到 $j$，笔试需统一方向定义，通常 `transitions[to_tag, from_tag]`）；
2. **硬约束设为极小负值**：
   - 绝不可能转移到 `<START>`；
   - 绝不可能从 `<STOP>` 发生转移；
   - `O` 不能直接转移到 `I-PER`（非法 BIO 序列）；
   - 将上述非法转移概率设为 $-10000.0$，在指数运算后概率恒等于 0。

In [ ]:
def init_transition_matrix(num_tags, tag_to_idx):
    """
    初始化转移矩阵并赋予物理硬约束
    transitions[to_tag, from_tag]
    """
    transitions = nn.Parameter(torch.randn(num_tags, num_tags))
    
    # 强制约束：绝不可能转移到 START
    transitions.data[tag_to_idx[START_TAG], :] = -10000.0
    # 强制约束：绝不可能从 STOP 转移到任何标签
    transitions.data[:, tag_to_idx[STOP_TAG]] = -10000.0
    
    # 语言学约束：O 绝不能直接跨越到 I-PER 或 I-LOC (必须先有 B-)
    transitions.data[tag_to_idx["I-PER"], tag_to_idx["O"]] = -10000.0
    transitions.data[tag_to_idx["I-LOC"], tag_to_idx["O"]] = -10000.0
    # I-PER 也不能直接跳到 I-LOC
    transitions.data[tag_to_idx["I-LOC"], tag_to_idx["I-PER"]] = -10000.0
    return transitions

# 测试验证约束
trans = init_transition_matrix(NUM_TAGS, TAG_TO_IDX)
assert trans[TAG_TO_IDX["I-PER"], TAG_TO_IDX["O"]].item() < -9000
print(">>> 转移矩阵初始化成功，非法转移已施加 -10000.0 硬掩码！")

---
## 模块三：金标真实路径得分 (Gold Path Score) 向量化计算

### 【笔试考点与推导】
- 真实路径序列 $y = (y_1, y_2, \dots, y_T)$ 的总得分为：
  $$\text{Score}(x, y) = \sum_{t=1}^T P_{t, y_t} + \sum_{t=0}^T A_{y_{t+1}, y_t}$$
  其中 $y_0 = \text{START}, y_{T+1} = \text{STOP}$，$P$ 为发射矩阵，$A$ 为转移矩阵。

In [ ]:
def compute_gold_score(feats, tags, transitions, tag_to_idx):
    """
    计算真实标签序列在当前模型参数下的真实得分 (标量)
    feats: (seq_len, num_tags)
    tags: (seq_len,) 真实标签下标
    transitions: (num_tags, num_tags) [to_tag, from_tag]
    """
    score = torch.zeros(1)
    # 起始状态: START -> tags[0]
    score = score + transitions[tags[0], tag_to_idx[START_TAG]]
    
    # 累加各时间步的发射得分与中间步转移得分
    for i in range(len(tags) - 1):
        score = score + feats[i, tags[i]]                               # 发射分
        score = score + transitions[tags[i + 1], tags[i]]               # 转移分 tags[i] -> tags[i+1]
        
    # 最后一个时间步发射分以及 -> STOP 的转移分
    score = score + feats[-1, tags[-1]]
    score = score + transitions[tag_to_idx[STOP_TAG], tags[-1]]
    return score

# 测试验证
dummy_feats = torch.tensor([[2.0, 1.0, 0.1, 0.2, 0.1, -10.0, -10.0],
                            [0.1, 0.2, 0.1, 3.0, 0.2, -10.0, -10.0]])
dummy_tags = torch.tensor([TAG_TO_IDX["O"], TAG_TO_IDX["B-LOC"]], dtype=torch.long)
gold_score = compute_gold_score(dummy_feats, dummy_tags, trans, TAG_TO_IDX)
print("真实黄金路径得分:", gold_score.item())
assert not torch.isnan(gold_score)

---
## 模块四：前向算法与规范化因子 $\log Z(x)$ (Log-Sum-Exp 向量化)

### 【笔试考点与防溢出】
- 所有路径得分的对数总和：
  $$Z(x) = \sum_{\tilde{y}} \exp(\text{Score}(x, \tilde{y}))$$
- **雷区**：若直接算 $\sum \exp$ 极易浮点溢出 (NaN)！必须使用 **`torch.logsumexp`** 技巧：
  $$\log \sum_i \exp(x_i) = m + \log \sum_i \exp(x_i - m), \quad m = \max_i(x_i)$$

In [ ]:
def forward_log_sum_exp(feats, transitions, tag_to_idx):
    """
    动态规划前向算法计算 log Z(x)
    feats: (seq_len, num_tags)
    transitions: (num_tags, num_tags) [to_tag, from_tag]
    """
    num_tags = transitions.size(0)
    # 初始化前向向量: 初始处于 START 状态，其余设为 -10000.0
    forward_var = torch.full((1, num_tags), -10000.0)
    forward_var[0, tag_to_idx[START_TAG]] = 0.0
    
    for feat in feats:
        # 矩阵广播技巧:
        # forward_var: (1, num_tags) -> 广播为 (num_tags, num_tags) [to, from]
        # feat: (num_tags,) -> 广播为 (num_tags, 1) 发射到每个目标标签
        # transitions: (num_tags, num_tags)
        emit_score = feat.view(-1, 1)                                   # (num_tags, 1)
        next_tag_var = forward_var + transitions + emit_score           # (num_tags, num_tags)
        
        # 对 from_tag 维度求 logsumexp
        forward_var = torch.logsumexp(next_tag_var, dim=1).view(1, -1)  # (1, num_tags)
        
    # 转移到 STOP 状态
    terminal_vars = forward_var + transitions[tag_to_idx[STOP_TAG]].view(1, -1)
    alpha = torch.logsumexp(terminal_vars, dim=1)
    return alpha

# 测试验证 log Z(x)
log_Z = forward_log_sum_exp(dummy_feats, trans, TAG_TO_IDX)
print("规范化常数 log Z(x):", log_Z.item())
assert log_Z.item() >= gold_score.item(), "数学定理: log Z(x) 必须大于等于任何单一路径得分!"
print(">>> 测试通过: log Z(x) >= Gold Score 严格满足！")

---
## 模块五：CRF 负对数似然损失 (CRF NLL Loss) 手撕

### 【笔试考点】
- CRF 损失函数闭式：
  $$\mathcal{L}_{\text{CRF}} = - \log P(y \mid x) = - \left( \text{Score}(x, y) - \log Z(x) \right) = \log Z(x) - \text{Score}(x, y)$$
- **优化物理含义**：压制所有候选伪路径的总能量 $\log Z(x)$，同时单独抬高真实路径的得分 $\text{Score}(x, y)$。

In [ ]:
def crf_loss(feats, tags, transitions, tag_to_idx):
    """
    计算单序列 CRF 负对数似然损失
    """
    forward_score = forward_log_sum_exp(feats, transitions, tag_to_idx)
    gold_score = compute_gold_score(feats, tags, transitions, tag_to_idx)
    return forward_score - gold_score

loss = crf_loss(dummy_feats, dummy_tags, trans, TAG_TO_IDX)
print(f"CRF NLL Loss 标量: {loss.item():.4f}")
assert loss.item() >= 0.0, "负对数似然损失必定非负!"
print(">>> 测试通过: 损失值非负，反向传播通路健全！")

---
## 模块六：维特比动态规划全局最优解码 (Viterbi Decoding with Backpointers)

### 【笔试考点与陷阱】
1. **与前向算法区别**：前向算法做 `logsumexp`（软和），维特比算法做 `max`（硬挑最优），并记录回溯指针 `backpointers`；
2. **回溯顺序**：必须从 `<STOP>` 节点找到最佳终点，再倒序（Reverse）一路追溯至第一个时间步！

In [ ]:
def viterbi_decode(feats, transitions, tag_to_idx):
    """
    维特比解码找出全局最优标签序列
    返回: (best_score, best_path)
    """
    num_tags = transitions.size(0)
    backpointers = []
    
    # 初始化维特比变量
    viterbi_vars = torch.full((1, num_tags), -10000.0)
    viterbi_vars[0, tag_to_idx[START_TAG]] = 0.0
    
    for feat in feats:
        # viterbi_vars 广播: (num_tags, num_tags)
        # transitions[to, from]
        next_tag_var = viterbi_vars.repeat(num_tags, 1) + transitions
        # 对每一个 to_tag 找到最大的 from_tag
        max_vars, bptrs = torch.max(next_tag_var, dim=1)
        
        # 加上当前发射分
        viterbi_vars = (max_vars + feat).view(1, -1)
        backpointers.append(bptrs.tolist())
        
    # 转移到 STOP 状态
    terminal_vars = viterbi_vars + transitions[tag_to_idx[STOP_TAG]].view(1, -1)
    best_tag_id = torch.argmax(terminal_vars).item()
    best_score = terminal_vars[0, best_tag_id].item()
    
    # 回溯指针找路径
    best_path = [best_tag_id]
    for bptrs_t in reversed(backpointers):
        best_tag_id = bptrs_t[best_tag_id]
        best_path.append(best_tag_id)
        
    # 剔除起始的 START
    start = best_path.pop()
    assert start == tag_to_idx[START_TAG]
    best_path.reverse()
    return best_score, best_path

best_score, best_path = viterbi_decode(dummy_feats, trans, TAG_TO_IDX)
idx_to_tag = {v: k for k, v in TAG_TO_IDX.items()}
print("维特比最佳路径得分:", round(best_score, 4))
print("维特比解码最佳标签序列:", [idx_to_tag[idx] for idx in best_path])

---
## 模块七：工业级批处理与变长 Padding Mask 机制 (Batched CRF)

### 【笔试考点与踩坑】
- 真实 NLP 任务中一个 Batch 的句子长度参差不齐（如短句被补齐为 `<PAD>`）；
- **核心逻辑**：对于已经结束并被填充的 Pad 位置：
  1. **发射分数置零**；
  2. **隐状态保持上一步原样透传**（即通过 Mask 选择：`v = mask * next_v + (1 - mask) * prev_v`）。

In [ ]:
class BatchedCRF(nn.Module):
    def __init__(self, num_tags, start_idx, stop_idx):
        super().__init__()
        self.num_tags = num_tags
        self.start_idx = start_idx
        self.stop_idx = stop_idx
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))
        self.transitions.data[start_idx, :] = -10000.0
        self.transitions.data[:, stop_idx] = -10000.0

    def forward_alg(self, emissions, mask):
        """
        emissions: (seq_len, batch_size, num_tags)
        mask: (seq_len, batch_size) 1 为真实 token, 0 为 padding
        """
        seq_len, batch_size, _ = emissions.shape
        init_alphas = torch.full((batch_size, self.num_tags), -10000.0, device=emissions.device)
        init_alphas[:, self.start_idx] = 0.0
        forward_var = init_alphas

        for i in range(seq_len):
            emit_score = emissions[i].unsqueeze(2)               # (B, num_tags, 1)
            # forward_var: (B, 1, num_tags)
            # transitions: (num_tags, num_tags)
            t_score = forward_var.unsqueeze(1) + self.transitions + emit_score
            next_tag_var = torch.logsumexp(t_score, dim=2)      # (B, num_tags)
            
            # 使用 mask 融合: 若为 pad，则状态不变!
            mask_i = mask[i].unsqueeze(1)                        # (B, 1)
            forward_var = mask_i * next_tag_var + (1 - mask_i) * forward_var

        terminal_vars = forward_var + self.transitions[self.stop_idx].unsqueeze(0)
        return torch.logsumexp(terminal_vars, dim=1)            # (B,)

# 测试批处理 Mask
batched_crf = BatchedCRF(NUM_TAGS, TAG_TO_IDX[START_TAG], TAG_TO_IDX[STOP_TAG])
dummy_emissions = torch.randn(3, 2, NUM_TAGS) # Seq=3, Batch=2
dummy_mask = torch.tensor([[1.0, 1.0], [1.0, 1.0], [1.0, 0.0]]) # 样本2长度为2被Pad
batched_logZ = batched_crf.forward_alg(dummy_emissions, dummy_mask)
print("批处理两样本 log Z(x):", batched_logZ.detach().numpy())
assert batched_logZ.shape == (2,)

---
## 模块八：BiLSTM-CRF 极简端到端序列标注模型封装

### 【笔试考点】
- **底层分工**：BiLSTM 负责抽取全局时序语义特征并输出各时刻发射分数；CRF 负责学习标签之间的转移概率，消除非法标注。

In [ ]:
class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, tag_to_idx, embedding_dim, hidden_dim):
        super().__init__()
        self.tag_to_idx = tag_to_idx
        self.num_tags = len(tag_to_idx)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2, num_layers=1, bidirectional=True, batch_first=False)
        self.hidden2tag = nn.Linear(hidden_dim, self.num_tags)
        self.crf = BatchedCRF(self.num_tags, tag_to_idx[START_TAG], tag_to_idx[STOP_TAG])

    def get_lstm_emissions(self, sentences):
        embeds = self.embedding(sentences)                      # (seq_len, batch_size, embed_dim)
        lstm_out, _ = self.lstm(embeds)                         # (seq_len, batch_size, hidden_dim)
        emissions = self.hidden2tag(lstm_out)                   # (seq_len, batch_size, num_tags)
        return emissions

# 验证模型实例化与前向维度
model = BiLSTM_CRF(vocab_size=100, tag_to_idx=TAG_TO_IDX, embedding_dim=16, hidden_dim=32)
dummy_sents = torch.randint(0, 100, (4, 2)) # 长度4, 批大小2
emits = model.get_lstm_emissions(dummy_sents)
print("BiLSTM-CRF 发射特征维度 (L, B, Num_Tags):", emits.shape)
assert emits.shape == (4, 2, NUM_TAGS)
print(">>> 端到端模型组装验证成功！")

---
## 模块九：MEMM 局部归一化“标注偏置 (Label Bias)”物理复现与 CRF 破局验证

### 【笔试必问八股与硬核实证】
1. **MEMM 标注偏置成因**：MEMM 在每一个状态做局部 Softmax 归一化 $\sum_j P(s_j \mid s_i) = 1$。如果某个状态的出度分支较少（例如只能转移到一个状态），它的局部转移概率恒为 1，直接忽略了观测特征 $X$ 的打分！
2. **CRF 全局归一化破解**：只在最后整条路径求指数和除以 $Z(x)$，各节点不强行局部归一化，彻底根除了标注偏置。

In [ ]:
def memm_vs_crf_label_bias_demo():
    """
    用极简玩具模型实证标注偏置
    假设有两个状态: State 1 (出度唯一, 只能到 State A), State 2 (出度发散, 可到 State A 或 B)
    """
    # MEMM 局部归一化
    # State 1 出度唯一 -> 局部概率为 1.0 (不管观测到什么!)
    p_memm_state1_next = np.array([1.0])
    # State 2 出度为 2 -> 均分概率 0.5
    p_memm_state2_next = np.array([0.5, 0.5])
    
    print("【MEMM 局部归一化死穴】:")
    print("  State 1 转移概率恒为 1.0, 即使底层特征发射打分极低, 也被局部归一化保送!")
    print("  State 2 转移概率被切分为 0.5, 哪怕其观测特征得分极高, 也会被分母强行稀释。")
    
    # CRF 全局归一化
    # 路径 1: score = emit1 + trans1 = -2.0 + 0.0 = -2.0
    # 路径 2: score = emit2 + trans2 = 10.0 + 0.0 = 10.0
    score_path1 = -2.0
    score_path2 = 10.0
    Z = np.exp(score_path1) + np.exp(score_path2)
    p_crf_path1 = np.exp(score_path1) / Z
    p_crf_path2 = np.exp(score_path2) / Z
    
    print("\n【CRF 全局归一化破局】:")
    print(f"  CRF 路径 1 全局概率: {p_crf_path1:.6f} (被低分特征正确淘汰)")
    print(f"  CRF 路径 2 全局概率: {p_crf_path2:.6f} (以 99.999% 胜出)")

memm_vs_crf_label_bias_demo()

---
## 企业笔试手撕核心口诀与雷区速记卡

```
1. 规范常数防溢出: 坚决使用 torch.logsumexp，严禁手动 sum(exp(x)) 导致 NaN！
2. 真实得分两步走: 既要累加各时刻发射分 emit[t, tag]，也要累加相邻时刻转移分 trans[next, cur]。
3. 维特比回溯避坑: 必须从 STOP 开始反向寻找指针，并最后 reverse 反转还原正序。
4. 批处理变长 Pad: 对 Padding 部分发射分置零，前向变量直接透传上一时刻值。
5. 标注偏置与全局归一: MEMM 局部归一化使稀疏分支保送，CRF 全局路径比拼消除偏差。
```